# Aarki Data Analyst Portfolio Project

## Executive Summary

This analysis addresses three questions. First, evaluated lead quality fell sharply from April through July 2009 and then partially recovered in August and September. Month-to-month differences are statistically significant overall, and the April to July decline is significant when tested directly. Second, time and debt segment are the clearest stable adjusted drivers of lead quality. Ad size and design also matter in the univariate screening, while partner source and campaign theme remain directionally useful but weaker after adjustment. Third, the economics in the prompt are favorable if lead quality rises from 8.0 percent to 9.6 percent, and the dataset points to the clearest opportunities in weak Brand, Google, and `302252` pockets. A full 20 percent lift still looks like an ambitious target that would need testing rather than a promise.

## Dataset Overview

| Item | Details |
|---|---|
| Primary dataset name | `Analyst_case_study_dataset` |
| Public data status | Raw lead-level data is not included in this public portfolio version. |
| Workbook structure | The workbook contains three sheets, `Sheet1`, `Sheet2`, and `Sheet3`. Only `Sheet1` contains data. |
| Observed grain | Each row represents one lead record with marketing context and call outcome information. |

### Key fields observed in the workbook

#### Lead and Contact Metadata
| Column Name | Description |
|---|---|
| `LeadCreated` | Timestamp when the lead record was created. |
| `FirstName` | Lead first name. This field was treated as sensitive metadata and excluded from modeling. |
| `Email` | Lead email address. This field was treated as sensitive metadata and excluded from modeling. |
| `VendorLeadID` | Unique lead identifier from the source or vendor system. This field was used only for ID checks. |
| `State` | U.S. state tied to the lead. |

#### Operational Status and Quality
| Column Name | Description |
|---|---|
| `CallStatus` | Outcome or status of the lead calling workflow. |
| `AddressScore` | Address quality or verification score. |
| `PhoneScore` | Phone quality or verification score. |
| `DebtLevel` | Reported debt amount bucket for the lead. |

#### Marketing Attribution
| Column Name | Description |
|---|---|
| `Partner` | Traffic partner or source delivering the lead. |
| `ReferralDomain` | Referring domain where the visit originated. |
| `MarketingCampaign` | Campaign label used in acquisition tracking. |
| `AdGroup` | Ad group label under the campaign. |
| `Keyword` | Paid search keyword associated with the lead. |
| `SearchQuery` | User search query text. |

#### Source and Landing Tracking
| Column Name | Description |
|---|---|
| `ReferralURL` | Full URL of the referring page or source. |
| `ReferralURL Parameters` | Query parameters captured from the referral URL. |
| `LandingPageURL` | Landing page URL visited before conversion. |
| `Landing Page URL Parameters` | Query parameters captured from the landing page URL. |

#### Publisher and Inventory Metadata
| Column Name | Description |
|---|---|
| `WidgetName` | Widget or placement identifier used for lead capture. |
| `PublisherZoneName` | Publisher zone or placement name. |
| `PublisherCampaignName` | Publisher side campaign name. |
| `AdvertiserCampaignName` | Advertiser side campaign name. |

### Three Research Questions We Will Answer
1. Are we seeing any lead quality trends over time, and are those trends statistically significant?
2. What can we learn about the drivers of lead quality from this dataset? Which segments, including where the ad was shown, what kind of person filled out the ad, and what kind of ad they saw, have different lead quality rates?
3. If the advertiser increases CPL by 20 percent, from $30 to $33, when lead quality increases by 20 percent, from 8.0 percent to 9.6 percent, do we see opportunities to achieve that here? What actions could support that improvement?


## 0. Dataset Description

This dataset contains information on leads generated through different advertising campaigns, with the goal of evaluating lead quality. The original dataset includes approximately 3,000 observations, where each observation represents an individual lead. It contains variables related to when the lead was created, where the ad was shown, the type and design of the advertisement, and characteristics of the individual, such as debt level.

The primary outcome variable is LeadQuality, which categorizes leads as Good, Bad, or Unknown based on call status. For modeling purposes, the analysis focuses on evaluated leads, meaning Good vs. Bad. The raw lead-level file is withheld from this public repository because it contains contact-level fields.

The dataset includes a mix of categorical variables (e.g., Partner, CampaignGroup, AdSize, FormType, Design, Color) and grouped variables (e.g., DebtLevelGrouped), allowing us to examine how different segments and ad characteristics are associated with lead quality. Overall, the dataset is designed to identify key drivers of lead quality across different advertising strategies and audience segments.

## 1. Setup and Packages


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import censusdata
import itertools
from pathlib import Path
from IPython.display import display
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
from scipy.stats import chi2
from statsmodels.stats.outliers_influence import variance_inflation_factor


## 1.5 Functions

In [ ]:
def lead_quality_table(df, column):
    full_count = pd.crosstab(df[column], df["LeadQuality"])
    full_pct = pd.crosstab(df[column], df["LeadQuality"], normalize="index") * 100
    full_table = full_count.astype(str) + " (" + full_pct.round(2).astype(str) + "%)"

    print(f"\nLeadQuality by {column} (all leads)")
    display(full_table)

    df_eval = df[df["LeadQuality"].isin(["Good", "Bad"])].copy()
    eval_count = pd.crosstab(df_eval[column], df_eval["LeadQuality"])
    eval_pct = pd.crosstab(df_eval[column], df_eval["LeadQuality"], normalize="index") * 100
    eval_table = eval_count.astype(str) + " (" + eval_pct.round(2).astype(str) + "%)"

    print(f"\nLeadQuality by {column} (evaluated leads)")
    display(eval_table)


def map_lead_quality(status):
    if status in ["Closed", "EP Sent", "EP Received", "EP Confirmed"]:
        return "Good"
    if status in [
        "Unable to contact - Bad Contact Information",
        "Contacted - Invalid Profile",
        "Contacted - Doesn't Qualify",
    ]:
        return "Bad"
    return "Unknown"


def combine_debt(x):
    if pd.isna(x):
        return np.nan
    if x == "7500-10000":
        return "<=10000"
    if x == "7500-15000":
        return "7500-15000"
    if x in ["10001-15000", "15001-20000"]:
        return "10001-20000"
    if x in ["20001-30000", "30001-50000"]:
        return "20001-50000"
    if x in ["50001-70000", "70001-90000"]:
        return "50001-90000"
    if x in ["90000-100000", "More_than_100000"]:
        return ">=90000"
    return np.nan


def group_campaign(campaign):
    if pd.isna(campaign):
        return "Other"

    if campaign in ["Credit", "Financial Services"]:
        return "Credit"

    if campaign in [
        "Debt",
        "Debt Consolidation",
        "Debt General",
        "Debt Holding Tank",
        "Debt Volume",
        "Debt Volume_CA",
        "Debt Volume_MI",
        "Debt Volume_TX",
        "Debt_Consolidation_-_SN",
        "Debt_General_-_SN",
    ]:
        return "Debt"

    if campaign in [
        "DebtReductionInc",
        "DebtReductionInc - YSM Restructure (05/29/09)",
    ]:
        return "Brand"

    return "Other"


def group_adgroup(x):
    if pd.isna(x):
        return "Other"

    x = x.lower()

    if "consolidation" in x:
        return "Debt Consolidation"
    if "settlement" in x or "settle" in x:
        return "Debt Settlement"
    if "bankruptcy" in x or "chapter" in x:
        return "Bankruptcy"
    if "credit card" in x:
        return "Credit Card Debt"
    if "student" in x:
        return "Student Debt"
    if "debt" in x:
        return "General Debt"
    return "Other"


def univariate_logistic(df, var):
    df_eval = df[df["LeadQuality"].isin(["Good", "Bad"])].copy()
    df_eval["GoodLead"] = (df_eval["LeadQuality"] == "Good").astype(int)
    df_eval = df_eval[df_eval[var].notna()].copy()

    model = smf.logit(f"GoodLead ~ C({var})", data=df_eval).fit()
    table = model.summary2().tables[1].copy()
    table.index = (
        table.index
        .str.replace(f"C({var})\\[T\\.", f"{var} = ", regex=True)
        .str.replace("]", "", regex=True)
    )

    if df_eval[var].nunique() > 2:
        mask = table.index != "Intercept"
        table.loc[mask, "Holm_p"] = multipletests(table.loc[mask, "P>|z|"], method="holm")[1]
    else:
        table["Holm_p"] = pd.NA

    print(f"\nUnivariate logistic regression: LeadQuality ~ {var}")
    print(f"Levels: {df_eval[var].nunique()}")
    print(f"LLR p-value: {model.llr_pvalue:.4g}\n")
    print(table)

    return model, table


def check_all_combinations_logit(df, outcome, predictors):
    rows = []

    for r in range(1, len(predictors) + 1):
        for combo in itertools.combinations(predictors, r):
            formula = f"{outcome} ~ " + " + ".join(f"C({var})" for var in combo)

            try:
                model = smf.logit(formula, data=df).fit(disp=False)
                rows.append({
                    "Num_Variables": r,
                    "Variables": combo,
                    "Formula": formula,
                    "Status": "Success",
                    "AIC": model.aic,
                    "BIC": model.bic,
                    "LLF": model.llf,
                })
            except Exception as e:
                rows.append({
                    "Num_Variables": r,
                    "Variables": combo,
                    "Formula": formula,
                    "Status": f"Failed: {type(e).__name__}",
                    "AIC": None,
                    "BIC": None,
                    "LLF": None,
                })

    return pd.DataFrame(rows)


def add_lrt_tests(results_df, predictors):
    results_df = results_df.copy()
    allowed = set(predictors)

    results_df["LRT_pvalue"] = pd.NA
    results_df["LRT_removed_var"] = pd.NA
    results_df["LRT_df_diff"] = pd.NA
    results_df["LRT_stat"] = pd.NA

    success_df = results_df[
        (results_df["Status"] == "Success")
        & (results_df["Variables"].apply(lambda x: set(x).issubset(allowed)))
    ].copy()

    lookup = {
        tuple(sorted(row["Variables"])): row
        for _, row in success_df.iterrows()
    }

    for i, row in success_df.iterrows():
        full_vars = tuple(sorted(row["Variables"]))
        full_ll = row["LLF"]
        full_df = row["Num_Variables"]

        best_p = None
        best_removed = None
        best_df_diff = None
        best_stat = None

        for var in full_vars:
            reduced_vars = tuple(v for v in full_vars if v != var)

            if reduced_vars not in lookup:
                continue

            reduced_row = lookup[reduced_vars]
            stat = 2 * (full_ll - reduced_row["LLF"])
            df_diff = full_df - reduced_row["Num_Variables"]
            p_value = chi2.sf(stat, df_diff)

            if best_p is None or p_value > best_p:
                best_p = p_value
                best_removed = var
                best_df_diff = df_diff
                best_stat = stat

        results_df.loc[i, "LRT_pvalue"] = best_p
        results_df.loc[i, "LRT_removed_var"] = best_removed
        results_df.loc[i, "LRT_df_diff"] = best_df_diff
        results_df.loc[i, "LRT_stat"] = best_stat

    return results_df


def check_logit_assumptions(model, data, outcome, categorical_vars, vif_threshold=10, prob_threshold=0.01, min_events_per_param=10):
    checks = []
    y = data[outcome]

    checks.append({
        "Check": "Binary outcome",
        "Passed": set(y.dropna().unique()).issubset({0, 1}) and y.nunique() == 2,
        "Details": f"Observed values: {sorted(y.dropna().unique().tolist())}",
    })

    converged = bool(model.mle_retvals.get("converged", False))
    checks.append({
        "Check": "Model convergence",
        "Passed": converged,
        "Details": f"Converged = {converged}",
    })

    X = model.model.exog
    rank = np.linalg.matrix_rank(X)
    checks.append({
        "Check": "Full-rank design matrix",
        "Passed": rank == X.shape[1],
        "Details": f"Rank = {rank}, Columns = {X.shape[1]}",
    })

    events = int(y.sum())
    non_events = int((1 - y).sum())
    events_per_param = min(events, non_events) / len(model.params)
    checks.append({
        "Check": "Adequate events per parameter",
        "Passed": events_per_param >= min_events_per_param,
        "Details": f"Events per parameter = {events_per_param:.2f}",
    })

    vif_rows = []
    for i, term in enumerate(model.model.exog_names):
        if term == "Intercept":
            continue
        vif_rows.append({
            "Term": term,
            "VIF": float(variance_inflation_factor(X, i)),
        })

    vif_table = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False).reset_index(drop=True)
    max_vif = vif_table["VIF"].max() if not vif_table.empty else np.nan
    high_vif = vif_table.loc[vif_table["VIF"] > vif_threshold, "Term"].tolist() if not vif_table.empty else []
    checks.append({
        "Check": "No severe multicollinearity",
        "Passed": len(high_vif) == 0,
        "Details": f"Max VIF = {max_vif:.2f}. High-VIF terms: {high_vif if high_vif else 'None'}",
    })

    fitted = model.predict()
    extreme = int(((fitted < prob_threshold) | (fitted > 1 - prob_threshold)).sum())
    checks.append({
        "Check": "No extreme fitted probabilities",
        "Passed": extreme == 0,
        "Details": f"Observations outside [{prob_threshold}, {1 - prob_threshold}] = {extreme}",
    })

    sparse_notes = []
    zero_cell_problem = False
    for var in categorical_vars:
        ct = pd.crosstab(data[var], y)
        zero_cells = int((ct == 0).sum().sum())
        min_cell = int(ct.min().min())
        sparse_notes.append(f"{var}: zero cells = {zero_cells}, min cell = {min_cell}")
        if zero_cells > 0:
            zero_cell_problem = True

    checks.append({
        "Check": "No zero outcome cells by predictor level",
        "Passed": not zero_cell_problem,
        "Details": ", ".join(sparse_notes),
    })

    return pd.DataFrame(checks), vif_table


def build_or_table(model):
    table = model.summary2().tables[1].copy()
    table["OR"] = np.exp(table["Coef."])
    table["CI_Lower_OR"] = np.exp(table["[0.025"])
    table["CI_Upper_OR"] = np.exp(table["0.975]"])
    return table[["Coef.", "P>|z|", "OR", "CI_Lower_OR", "CI_Upper_OR"]].round(3)

def q3_reallocation_scenario(frame, col, from_seg, to_seg, share, rate_col="closed_rate"):
    seg = q3_segment_summary(frame, col).set_index(col)
    move = share * seg.loc[from_seg, "leads"]
    current_total = (seg["leads"] * seg[rate_col]).sum()
    new_total = current_total + move * (seg.loc[to_seg, rate_col] - seg.loc[from_seg, rate_col])
    new_rate = new_total / seg["leads"].sum()
    return new_rate

## 2. Data Loading and Dataset Read


In [ ]:
data_path = Path("data/Analyst_case_study_dataset.xls")

if not data_path.exists():
    raise FileNotFoundError(
        "The raw dataset is not included in this public portfolio repository. "
        "Place the source file at data/Analyst_case_study_dataset.xls to rerun the notebook."
    )

df = pd.read_excel(data_path, sheet_name="Sheet1", engine="xlrd")

print("Shape:", df.shape)
sensitive_cols = ["FirstName", "Email", "VendorLeadID", "IP Address"]
df.drop(columns=sensitive_cols, errors="ignore").head()

## 3. Initial Data Quality Checks


### 3.1 Check for data missningness

Before performing any analysis, it is important to first examine the dataset. Fortunately, most of the data are quite complete, but some columns contain missing values. I therefore identify which columns have missing values and calculate the percentage of missing data in each column.


In [ ]:
missing = df.isna().sum()
missing_percent = (missing / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percent": missing_percent
})

missing_summary = missing_summary[missing_summary["Missing_Count"] > 0]
missing_summary.sort_values("Missing_Percent", ascending=False)

The table indicates that the IP Address column has 100% missing values, which aligns with the dataset description stating that IP addresses were removed. Several other variables also contain missing values. At this stage, we will simply document these missing values and revisit them later when determining appropriate data-cleaning strategies.


### 3.2 Check for Data Type

In [ ]:
df.dtypes

### 3.3 Check for Unique Values/Categorical

In [ ]:
summary_list = []

for col in df.columns:
    unique_vals = df[col].nunique(dropna=False)
    
    if unique_vals > 50:
        levels = "Text Data"
    else:
        levels = ", ".join(map(str, df[col].value_counts().index))
    
    summary_list.append({
        "Column": col,
        "Unique_Values": unique_vals,
        "Levels": levels
    })

summary_df = pd.DataFrame(summary_list)
summary_df

In [ ]:
pd.set_option('display.max_colwidth', None)
result_list = []

for col in df.columns:
    unique_vals = df[col].nunique(dropna=False)
    
    if unique_vals <= 50:
        unique_levels = ", ".join(map(str, df[col].drop_duplicates()))
        
        result_list.append({
            "Column": col,
            "Unique_Values": unique_vals,
            "Unique_Levels": unique_levels
        })

result_df = pd.DataFrame(result_list)

result_df

Here, I check the number of unique values in each column. I set a threshold of 50, meaning that any column with more than 50 unique values is classified as free-response text data. These columns may not be used in the analysis because they would require additional time and methods for text processing. After generating the results, I will manually review them to ensure that the unique value counts are correct. Furthermore, for columns that are not classified as text data, I will print out their unique values and manually inspect them to identify any typos or values with the same meaning so they can be combined if necessary. 

From this inspection, there are several issues to be aware of. First, the DebtLevel variable may need to be further combined because having too many levels could affect the results of later analyses. Second, some columns, such as Partner, require additional cleaning because values like "google" and "Google" represent the same category and should be combined. Third, the MarketingCampaign column is labeled as Campaign in the dataset description, so this difference should be noted to avoid confusion.

### 3.4 Duplicates

In [ ]:
df.duplicated().sum() #no duplicates row

### 3.5 Summary Statistics (numeric variables) and Outliers

In [ ]:
df.describe() #ignore the IP Address

In [ ]:
#check for numeric variable outliers
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.boxplot(y=df["AddressScore"])
plt.title("AddressScore Boxplot")

plt.subplot(1,2,2)
sns.boxplot(y=df["PhoneScore"])
plt.title("PhoneScore Boxplot")

plt.tight_layout()
plt.show()

In [ ]:
def count_outliers(series): #more formal test
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return ((series < lower) | (series > upper)).sum()

print("AddressScore outliers:", count_outliers(df["AddressScore"])) 
print("PhoneScore outliers:", count_outliers(df["PhoneScore"])) #no outlier

Through the earlier data examination, we have gained a basic understanding of the dataset. We will now focus on the variables that are relevant to the research questions. If additional cleaning is needed, I will return to this section to perform further data cleaning.

# Question 1: Are we seeing any lead quality trends over time (improving, declining)? Are they statistically significant?

To answer the question, we first need to identify which variables are relevant. Then, we can perform additional data cleaning based on those variables.

In [ ]:
# Define status groups using the exact raw CallStatus values
good_status = ["Closed", "EP Sent", "EP Received", "EP Confirmed"]
bad_status = [
    "Unable to contact - Bad Contact Information",
    "Contacted - Invalid Profile",
    "Contacted - Doesn't Qualify",
]

# Create LeadQuality variable
df["LeadQuality"] = df["CallStatus"].apply(map_lead_quality)

# Create summary table
lead_quality_summary = (
    df["LeadQuality"]
    .value_counts()
    .to_frame("Count")
    .assign(Percent=lambda x: (x["Count"] / x["Count"].sum() * 100).round(2))
)

lead_quality_summary


To simplify the analysis, I grouped the raw `CallStatus` field into `Good`, `Bad`, and `Unknown` using the exact source labels in the workbook. `Closed`, `EP Sent`, `EP Received`, and `EP Confirmed` are treated as `Good`. `Unable to contact - Bad Contact Information`, `Contacted - Invalid Profile`, and `Contacted - Doesn't Qualify` are treated as `Bad`. All remaining records are labeled `Unknown`.

With that corrected mapping, 70.84 percent of leads are `Unknown`, 13.01 percent are `Good`, and 16.15 percent are `Bad`. This updated definition increases the evaluated sample substantially and gives a more reliable base for Questions 1 and 2.


In [ ]:
#Column A: Lead created date
lead_created_summary = {
    "Earliest Date": df["LeadCreated"].min(),
    "Latest Date": df["LeadCreated"].max(),
    "Total Observations": df["LeadCreated"].count(),
    "Missing Values": df["LeadCreated"].isna().sum(),
    "Unique Days": df["LeadCreated"].dt.date.nunique()
}

pd.DataFrame([lead_created_summary])

In [ ]:
#Create the value for month for later time analysis 
df["month"] = df["LeadCreated"].dt.month
df[["LeadCreated","month"]].head()

In [ ]:
df["month"].value_counts().sort_index().plot(kind="line", marker="o")
plt.xlabel("Month")
plt.ylabel("Number of Leads")
plt.title("Number of Leads by Month")
plt.show()

In [ ]:
#Monthly Percentage by leadquality
monthly_quality = (
    df.groupby(["month", "LeadQuality"])
    .size()
    .unstack(fill_value=0)
)

monthly_percent = monthly_quality.div(monthly_quality.sum(axis=1), axis=0) * 100

import matplotlib.pyplot as plt

monthly_percent.plot(marker="o", figsize=(8,5))

plt.xlabel("Month")
plt.ylabel("Percentage (%)")
plt.title("Lead Quality Percentage by Month")
plt.legend(title="LeadQuality")
plt.show()

In [ ]:
#Column F WidgetName
df["WidgetName"].value_counts()

In [ ]:
#create new variable from widget_parts
widget_parts = df["WidgetName"].str.split("-", expand=True)

df["AdSize"] = widget_parts[1]
df["FormType"] = widget_parts[3]
df["Design"] = widget_parts[4]
df["Color"] = widget_parts[5]

In [ ]:
#formtype comparison
lead_quality_table(df, "FormType")

In [ ]:
#design comparison
lead_quality_table(df, "Design")

In [ ]:
#adsize comparison
lead_quality_table(df, "AdSize")

In [ ]:
#color comparison
lead_quality_table(df, "Color")

In [ ]:
#Column G PublisherZoneName
lead_quality_table(df, "PublisherZoneName")

In [ ]:
#Column H — PublisherCampaignName
lead_quality_table(df, "PublisherCampaignName")

In [ ]:
#Column I — AddressScore
lead_quality_table(df, "AddressScore")
#Since there are a lot of missingness, so we won't use this variable
#potential missingness bias 

In [ ]:
#Column J — PhoneScore
lead_quality_table(df, "PhoneScore")

In [ ]:
#Column K — AdvertiserCampaignName
lead_quality_table(df, "AdvertiserCampaignName")

In [ ]:
# Column M — Debt level
# Keep the overlapping 7500-15000 source bucket separate instead of forcing it into a higher band.
df["DebtLevelGrouped"] = df["DebtLevel"].apply(combine_debt)

order = ["<=10000", "7500-15000", "10001-20000", "20001-50000", "50001-90000", ">=90000"]

df["DebtLevelGrouped"] = pd.Categorical(
    df["DebtLevelGrouped"],
    categories=order,
    ordered=True,
)

lead_quality_table(df, "DebtLevelGrouped")


In [ ]:
#Column O — Partner
lead_quality_table(df, "Partner") #one level have very few count

In [ ]:
# Drop observations with extremely small category
df = df[df["Partner"] != "Advertise.com"].copy()

# Clean / group partner names
df["PartnerClean"] = df["Partner"].replace({
    "google": "Google",
    "Google": "Google",
    "Call_Center": "Call Center",
    "yahoo": "Yahoo"
})

# Generate the lead quality table
lead_quality_table(df, "PartnerClean")

In [ ]:
#Column Q — Campaign
df["CampaignGroup"] = df["MarketingCampaign"].apply(group_campaign)
lead_quality_table(df, "CampaignGroup")


The original MarketingCampaign variable contains many detailed campaign names, several with very small sample sizes. To simplify the analysis and avoid unstable estimates, campaigns were grouped into four broader categories based on their marketing theme: Credit, Debt, Brand, and Other. This grouping preserves meaningful differences in marketing strategy while improving interpretability.

In [ ]:
#Column R — AdGroup
df["AdGroupGroup"] = df["AdGroup"].apply(group_adgroup)

lead_quality_table(df, "AdGroupGroup")

The AdGroup variable contains many detailed keyword names and small categories. To simplify the analysis and reduce noise from sparse groups, similar keywords were grouped into broader themes such as Debt Consolidation, Credit Card Debt, Bankruptcy, General Debt, Student Debt, and Other based on search intent. After grouping, lead quality differs across categories: General Debt, Credit Card Debt, and Debt Consolidation produce the highest share of good leads (70–78%), while Student Debt produces substantially lower-quality leads (11% good).

The EDA above shows the count and percentage breakdowns for the variables that appear most relevant to the research questions. Domain experts could still highlight additional variables, and in a production setting I would review those choices with the business team before finalizing the analysis. For this assignment, I proceed with the variables explored here and exclude free response text fields that are not central to the research questions.


In [ ]:
df_q1 = df.loc[df["LeadCreated"].notna(), ["LeadCreated", "LeadQuality"]].copy()
df_q1["month_period"] = df_q1["LeadCreated"].dt.to_period("M").astype(str)
monthly_summary = (
    df_q1.groupby("month_period")
    .agg(
        Total_Leads=("LeadQuality", "size"),
        Good=("LeadQuality", lambda s: (s == "Good").sum()),
        Bad=("LeadQuality", lambda s: (s == "Bad").sum()),
        Unknown=("LeadQuality", lambda s: (s == "Unknown").sum())
    )
    .reset_index()
    .sort_values("month_period")
    .reset_index(drop=True)
)
monthly_summary

In [ ]:
monthly_summary["Evaluated_Leads"] = monthly_summary["Good"] + monthly_summary["Bad"]
monthly_summary["Good_Rate_Evaluated"] = np.where(
    monthly_summary["Evaluated_Leads"] > 0,
    monthly_summary["Good"] / monthly_summary["Evaluated_Leads"] * 100,
    np.nan
)
monthly_summary["Unknown_Rate_Total"] = (
    monthly_summary["Unknown"] / monthly_summary["Total_Leads"] * 100
)
print("Monthly summary:")
display(monthly_summary)

The monthly summary still shows a clear deterioration in evaluated lead quality through the middle of the period, but the corrected status mapping changes the magnitude. The good lead rate among evaluated leads falls from 58.1 percent in April to 50.4 percent in May, 37.9 percent in June, and a low of 30.8 percent in July. It then recovers to 48.1 percent in August and 50.9 percent in September. Unknown outcomes remain very large in every month, ranging from about 62.7 percent to 80.1 percent of all leads, so the time analysis should still be interpreted as a trend among evaluated leads rather than the full lead population.


In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=monthly_summary,
    x="month_period",
    y="Good_Rate_Evaluated",
    marker="o"
)

The figure shows a pronounced decline in the evaluated good rate from April through July 2009, followed by a partial recovery in August and September. The corrected mapping lowers the absolute levels compared with the earlier draft, but the overall shape remains the same: deterioration through July, then rebound.


In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=monthly_summary,
    x="month_period",
    y="Unknown_Rate_Total",
    marker="o",
    color="orange"
)

Unknown outcomes remain very high in every month, ranging from roughly 63 percent to 80 percent of total leads. That does not invalidate the evaluated-lead analysis, but it does mean that the monthly quality pattern should be interpreted with caution because outcome availability is uneven across time.


In [ ]:
df_eval = df_q1[df_q1["LeadQuality"].isin(["Good", "Bad"])].copy()
df_eval["is_good"] = (df_eval["LeadQuality"] == "Good").astype(int)
month_order = sorted(df_eval["month_period"].unique())
month_map = {m: i + 1 for i, m in enumerate(month_order)}
df_eval["month_num"] = df_eval["month_period"].map(month_map)
eval_counts = (
    df_eval.groupby("month_period")
    .agg(
        Evaluated_Leads=("is_good", "count"),
        Good_Leads=("is_good", "sum")
    )
    .reset_index()
)
eval_counts["Good_Rate"] = eval_counts["Good_Leads"] / eval_counts["Evaluated_Leads"] * 100
print("Evaluated leads by month:")
display(eval_counts)

Among evaluated leads, the corrected monthly good rates are 58.1 percent in April, 50.4 percent in May, 37.9 percent in June, 30.8 percent in July, 48.1 percent in August, and 50.9 percent in September. This supports a decline through July followed by a partial recovery, rather than one smooth monotonic trend across the full period.


In [ ]:
full_linear_model = sm.Logit(
    df_eval["is_good"],
    sm.add_constant(df_eval["month_num"]),
).fit(disp=False)

null_model = smf.logit("is_good ~ 1", data=df_eval).fit(disp=False)
month_model = smf.logit("is_good ~ C(month_period)", data=df_eval).fit(disp=False)

overall_lr_stat = 2 * (month_model.llf - null_model.llf)
overall_df_diff = month_model.df_model - null_model.df_model
overall_month_p = chi2.sf(overall_lr_stat, overall_df_diff)

df_eval_apr_jul = df_eval[df_eval["month_period"].isin(["2009-04", "2009-05", "2009-06", "2009-07"])].copy()
apr_jul_order = sorted(df_eval_apr_jul["month_period"].unique())
apr_jul_map = {m: i + 1 for i, m in enumerate(apr_jul_order)}
df_eval_apr_jul["month_num"] = df_eval_apr_jul["month_period"].map(apr_jul_map)

apr_jul_model = sm.Logit(
    df_eval_apr_jul["is_good"],
    sm.add_constant(df_eval_apr_jul["month_num"]),
).fit(disp=False)

q1_test_summary = pd.DataFrame({
    "Test": [
        "Overall month differences, month as categorical",
        "April to July decline, linear month index",
        "Full April to September linear month index",
    ],
    "Coefficient": [
        np.nan,
        round(apr_jul_model.params["month_num"], 4),
        round(full_linear_model.params["month_num"], 4),
    ],
    "p_value": [
        overall_month_p,
        apr_jul_model.pvalues["month_num"],
        full_linear_model.pvalues["month_num"],
    ],
})

print("Question 1 test summary")
display(q1_test_summary.round(4))

print("Month-as-categorical model")
display(month_model.summary2().tables[1].round(4))


The statistical tests are more informative when they match the shape of the data. A logistic model that treats month as a categorical variable shows that month-to-month differences in evaluated lead quality are significant overall (`LR p < 0.001`). A second test that focuses on the deterioration phase only, April through July, also shows a significant negative linear trend (`β = -0.394`, `p < 0.001`). By contrast, a single linear month index across the full April to September period is not significant (`p = 0.137`) because lead quality rebounds after July. For Question 1, the most accurate interpretation is therefore a sharp decline through July followed by partial recovery, not one continuous linear decline across the full window.


## Answer to Question 1

To evaluate whether lead quality changed over time, I grouped `CallStatus` into `Good`, `Bad`, and `Unknown` using the exact raw status labels in the workbook. Because many leads do not have a definitive final outcome, the main trend analysis focuses on evaluated leads only, defined as records classified as either `Good` or `Bad`. Lead creation dates were then aggregated to the monthly level using `LeadCreated`.

With the corrected mapping, the monthly pattern shows a clear deterioration through July 2009 followed by a partial recovery. Among evaluated leads, the good lead rate falls from 58.1 percent in April to 50.4 percent in May, 37.9 percent in June, and 30.8 percent in July. It then improves to 48.1 percent in August and 50.9 percent in September. The descriptive pattern is therefore best summarized as decline through July with recovery afterward.

To test whether the changes over time are statistically meaningful, I used two complementary specifications. A logistic model with month treated as a categorical predictor shows that lead quality differs significantly across months overall (`LR p < 0.001`). I then fit a linear month trend restricted to April through July, which matches the deterioration phase of the data. That test is also significant (`β = -0.394`, `p < 0.001`), confirming that lead quality declined materially during that period. By contrast, a single linear trend across the full April to September window is not significant (`p = 0.137`) because the quality rebound in August and September breaks the linear pattern.

Unknown outcomes still account for about 63 percent to 80 percent of total leads in each month, so the analysis reflects trends among leads with observed outcomes rather than the full lead population. Even with that limitation, the evidence supports the conclusion that lead quality worsened substantially from April through July 2009 and then partially recovered, with statistically significant month-to-month differences overall.


# Question 2: What can we learn about the drivers of "lead quality" from this dataset? What segments - where the ad was shown, what kind of person filled out the ad, what kind of ad did they see - have differing lead quality rates?

The earlier EDA highlights several variables that may help explain differences in lead quality. These variables describe where the ad appeared, what kind of ad the user saw, and characteristics of the person who submitted the form.

I first use univariate logistic regression as an exploratory screening step with `Good` versus `Bad` leads as the outcome. At this stage, I use a relaxed threshold of `p < 0.20` to flag variables that may still be worth carrying forward.

Before fitting the adjusted model, I also examine variable coverage and overlap. Some predictors retain very little data, while others capture nearly the same information. I keep month in the adjusted model because Question 1 showed that time matters, and I compare the final multivariable models on a common analysis sample.


In [ ]:
model, table = univariate_logistic(df, "AdSize")


The univariate logistic regression shows that lead quality differs significantly by `AdSize` (`LLR p = 0.022`). Using `300250` as the reference, `302252` has significantly lower odds of producing a good evaluated lead (`p = 0.022`). This makes ad size one of the clearer creative signals in the Question 2 screening step.


In [ ]:
model, table = univariate_logistic(df, "FormType")


The univariate logistic regression does not show meaningful differences by `FormType` after the corrected lead-quality mapping (`LLR p = 0.577`). In this version of the data, form type does not appear to be a strong standalone driver of evaluated lead quality.


In [ ]:
model, table = univariate_logistic(df, "Design")


The univariate logistic regression still shows that `Design` matters (`LLR p = 0.025`), although the usable sample is smaller because of missing values. Relative to the reference design, `CreditSolutions`, `white`, and `yellowarrow` all have significantly lower odds of generating a good evaluated lead after Holm correction. This suggests that creative design remains a meaningful signal, but it should be interpreted more cautiously than full-coverage variables.


In [ ]:
model, table = univariate_logistic(df, "Color")


The univariate logistic regression shows no evidence that `Color` is an important standalone driver of evaluated lead quality (`LLR p = 0.894`). Color remains a weak signal and is not a strong candidate for the final adjusted model.


In [ ]:
model, table = univariate_logistic(df, "PublisherZoneName")


After correcting the lead-quality mapping, `PublisherZoneName` no longer shows a statistically significant univariate association with evaluated lead quality (`LLR p = 0.303`). Placement may still matter in combination with other variables, but it is not a strong standalone signal here.


In [ ]:
model, table = univariate_logistic(df, "PublisherCampaignName")


`PublisherCampaignName` now tells the same story as `PublisherZoneName`: the univariate association is not statistically significant (`LLR p = 0.303`). Because the two variables are also highly overlapping, neither one becomes a strong adjusted candidate on its own.


In [ ]:
model, table = univariate_logistic(df, "DebtLevelGrouped")


`DebtLevelGrouped` remains one of the strongest signals in the notebook (`LLR p < 0.001`). Relative to the lowest debt bucket, the overlapping `7500-15000` bucket and the clearer mid-debt buckets all have significantly higher odds of producing a good evaluated lead, while the highest bucket, `>=90000`, is not statistically significant after correction. Debt level is therefore still the clearest person-level driver in the analysis.


In [ ]:
model, table = univariate_logistic(df, "PartnerClean")


`PartnerClean` shows a weaker but still meaningful overall association with evaluated lead quality (`LLR p = 0.048`). Call Center and Google both perform worse than AdKnowledge in the raw comparison, but those pairwise differences do not remain significant after Holm correction. I therefore treat partner source as directionally informative and worth controlling for, rather than as the single clearest independent driver.


In [ ]:
model, table = univariate_logistic(df, "CampaignGroup")


`CampaignGroup` shows only weak exploratory evidence of association in the univariate screen (`LLR p = 0.166`). Credit campaigns look directionally stronger than Brand campaigns, but the grouped campaign effect is not strong enough on its own to be treated as a definitive standalone driver.


### Why I changed the multivariable strategy

In the earlier version of the notebook, I created one large predictor list and compared many possible logistic regression combinations using BIC and likelihood ratio tests. The goal was to let the data determine which subset of variables best explained lead quality.

That approach becomes problematic in this dataset for several reasons. Different formulas use different implicit samples because `statsmodels` drops missing rows separately for each model. Some predictors also overlap heavily. `PublisherCampaignName` is essentially a relabeling of `PublisherZoneName`, and `AdSize` overlaps strongly with `FormType`. In addition, some variables shrink the usable sample sharply. `Design` has meaningful missingness, and `Color` is observed for only a small fraction of evaluated leads. The earlier all-combinations search therefore produced an implausible result, where a very small model centered on `Color` appeared to perform best even though that conclusion did not match the univariate results or the broader EDA.

To address this, I switched to a staged common-sample approach. I separate predictors into forced variables and candidate variables. I keep `month_period` in the model because Question 1 already showed that time matters. I then compare candidate variables one at a time on top of the same adjusted base model, using the same rows for every staged comparison.

This is also why the predictor list now looks smaller. The original list captured everything that might matter. The revised structure keeps the variables that can be compared fairly in a stable adjusted model. Variables that remain useful but unstable are still discussed in the EDA and univariate sections rather than being forced into the final multivariable model.


In [ ]:
if "month_period" not in df.columns:
    df["month_period"] = df["LeadCreated"].dt.to_period("M").astype(str)

df_eval_q2 = df[df["LeadQuality"].isin(["Good", "Bad"])].copy()
df_eval_q2["GoodLead"] = (df_eval_q2["LeadQuality"] == "Good").astype(int)

q2_vars = [
    "month_period",
    "AdSize",
    "FormType",
    "Design",
    "Color",
    "PublisherZoneName",
    "PublisherCampaignName",
    "DebtLevelGrouped",
    "PartnerClean",
    "CampaignGroup",
]

q2_coverage = pd.DataFrame({
    "Non_Missing": [df_eval_q2[var].notna().sum() for var in q2_vars],
    "Missing": [df_eval_q2[var].isna().sum() for var in q2_vars],
    "Levels": [df_eval_q2[var].dropna().nunique() for var in q2_vars],
}, index=q2_vars)

q2_coverage["Coverage_%"] = (q2_coverage["Non_Missing"] / len(df_eval_q2) * 100).round(1)

print(f"Evaluated leads: {len(df_eval_q2)}")
display(q2_coverage.sort_values(["Coverage_%", "Levels"], ascending=[False, False]))


The coverage table shows why the final adjusted model needs to be more selective than the univariate screening step. Variables such as `month_period`, `AdSize`, `PublisherZoneName`, `DebtLevelGrouped`, `PartnerClean`, and `CampaignGroup` retain the full evaluated sample, while `Design` drops a meaningful number of rows and `Color` retains only a very small subset of the evaluated leads.

Because of that, forcing all candidate variables into one logistic model would shrink the usable sample sharply and can make the final result depend more on missingness than on genuine lead-quality differences.


In [ ]:
print("Publisher zone vs. publisher campaign")
display(pd.crosstab(df_eval_q2["PublisherZoneName"], df_eval_q2["PublisherCampaignName"]))

print("AdSize vs. FormType")
display(pd.crosstab(df_eval_q2["AdSize"], df_eval_q2["FormType"]))


These cross-tabulations show that some candidate predictors overlap heavily. `PublisherCampaignName` is essentially a relabeling of `PublisherZoneName` in this dataset, so keeping both in the same adjusted model would be redundant. `AdSize` and `FormType` also overlap strongly, so I compare them carefully rather than forcing every creative field into the same final specification.

Based on these checks, I exclude `PublisherCampaignName` from the adjusted model, keep `month_period` as a control, and use a staged model-building approach for the remaining higher-coverage variables.


In [ ]:
forced_vars = ["month_period", "DebtLevelGrouped", "PartnerClean"]
candidate_vars = ["CampaignGroup", "AdSize", "FormType"]

df_q2_model = df_eval_q2[["GoodLead", *forced_vars, *candidate_vars]].dropna().copy()

base_formula = "GoodLead ~ C(month_period) + C(DebtLevelGrouped) + C(PartnerClean)"
model_specs = {
    "Base model": base_formula,
    "Base + CampaignGroup": base_formula + " + C(CampaignGroup)",
    "Base + AdSize": base_formula + " + C(AdSize)",
    "Base + FormType": base_formula + " + C(FormType)",
}

models = {}
rows = []

for name, formula in model_specs.items():
    fit = smf.logit(formula, data=df_q2_model).fit(disp=False)
    models[name] = fit

base_model = models["Base model"]

for name, fit in models.items():
    row = {
        "Model": name,
        "AIC": round(fit.aic, 2),
        "BIC": round(fit.bic, 2),
        "LRT_vs_Base_p": np.nan,
    }

    if name != "Base model":
        stat = 2 * (fit.llf - base_model.llf)
        df_diff = fit.df_model - base_model.df_model
        row["LRT_vs_Base_p"] = round(chi2.sf(stat, df_diff), 4)

    rows.append(row)

comparison_table = pd.DataFrame(rows)

best_bic_label = comparison_table.sort_values("BIC").iloc[0]["Model"]
best_bic_formula = model_specs[best_bic_label]
best_bic_model = models[best_bic_label]

best_lrt_label = (
    comparison_table
    .dropna(subset=["LRT_vs_Base_p"])
    .sort_values("LRT_vs_Base_p")
    .iloc[0]["Model"]
)
best_lrt_formula = model_specs[best_lrt_label]
best_lrt_model = models[best_lrt_label]

display(comparison_table)


The staged comparison shows that BIC and the exploratory likelihood-ratio screen do not point to exactly the same expanded model. BIC favors the simpler base model, while the LRT screen suggests that `CampaignGroup` is the only additional variable with borderline incremental value.

Because model selection and model adequacy are different questions, the next step is to inspect the two candidate models directly and then run model diagnostics before deciding which one is stable enough to report as the final adjusted model.


In [ ]:
print(f"Best by BIC: {best_bic_label}")
print(best_bic_formula)
display(best_bic_model.summary2())
display(build_or_table(best_bic_model))

print(f"Best by LRT: {best_lrt_label}")
print(best_lrt_formula)
display(best_lrt_model.summary2())
display(build_or_table(best_lrt_model))


### Results Interpretation

The staged comparison identifies two candidate adjusted models worth discussing. BIC selects the simpler base model with `month_period`, `DebtLevelGrouped`, and `PartnerClean`. The exploratory LRT screen instead points to the expanded model that adds `CampaignGroup`.

That does not automatically mean the LRT-expanded model should become the final reporting model. Before making that decision, it is important to check whether both models satisfy the core practical assumptions needed for a stable logistic regression fit. In this notebook, I focus on convergence, full-rank design matrix, event-per-parameter ratio, extreme fitted probabilities, sparse cells, and multicollinearity.


### Best-Model Summaries and Assumption Checks

The next cells print the two candidate adjusted models, apply the diagnostic function to each one, and then keep the cleaner model as the final reporting specification if the expanded model fails the checks.


In [ ]:
bic_assumptions, bic_vif = check_logit_assumptions(
    best_bic_model,
    df_q2_model,
    outcome="GoodLead",
    categorical_vars=forced_vars,
)

lrt_assumptions, lrt_vif = check_logit_assumptions(
    best_lrt_model,
    df_q2_model,
    outcome="GoodLead",
    categorical_vars=forced_vars + ["CampaignGroup"],
)

print("BIC model checks")
display(bic_assumptions)
display(bic_vif.head(10).round(2))

print("LRT model checks")
display(lrt_assumptions)
display(lrt_vif.head(10).round(2))


In [ ]:
if bic_assumptions["Passed"].all():
    print("The BIC model passes the checks.")
else:
    print("The BIC model still has unresolved issues.")

if lrt_assumptions["Passed"].all():
    final_q2_reporting_label = best_lrt_label
    final_q2_reporting_formula = best_lrt_formula
    final_q2_reporting_model = best_lrt_model
    print("The LRT model also passes the checks.")
else:
    failed_checks = lrt_assumptions.loc[~lrt_assumptions["Passed"], "Check"].tolist()
    print("The LRT model fails these checks:", failed_checks)
    print("I drop CampaignGroup here because it overlaps too much with PartnerClean.")
    final_q2_reporting_label = best_bic_label
    final_q2_reporting_formula = best_bic_formula
    final_q2_reporting_model = best_bic_model

print(f"Final reporting model: {final_q2_reporting_label}")
print(final_q2_reporting_formula)
display(final_q2_reporting_model.summary2())
display(build_or_table(final_q2_reporting_model))


The diagnostics show that the BIC-best model is well behaved: it converges, has a full-rank design matrix, has no extreme fitted probabilities, no zero outcome cells, and only moderate VIF values. The LRT-best model also converges, but it fails the multicollinearity check because `CampaignGroup` overlaps too strongly with `PartnerClean`, which inflates several VIF values above the threshold.

Because `CampaignGroup` only adds weak exploratory signal and creates an instability problem in the adjusted model, the clean fix is to drop it from the final reporting specification. That leaves the simpler BIC-best model as the final adjusted model for Question 2.


### Final Question 2 Model Choice

| Criterion | Model or result |
|---|---|
| Best by `BIC` | `GoodLead ~ C(month_period) + C(DebtLevelGrouped) + C(PartnerClean)` |
| Best by exploratory `LRT` | `GoodLead ~ C(month_period) + C(DebtLevelGrouped) + C(PartnerClean) + C(CampaignGroup)` |
| Diagnostic outcome | The LRT expanded model fails the multicollinearity check. |
| Final reporting model | `GoodLead ~ C(month_period) + C(DebtLevelGrouped) + C(PartnerClean)` |

The final adjusted takeaway is that time and debt segment are the most stable drivers of lead quality in this dataset. `PartnerClean` remains in the model as an important control variable. `CampaignGroup` still provides useful exploratory insight, but it overlaps too strongly with partner mix to be treated as a stable independent predictor once partner source is already in the model.


## Answer to Question 2

To understand the drivers of lead quality, I first examined segment-level differences using EDA tables and univariate logistic regressions on corrected evaluated leads (`Good` vs. `Bad`). In that exploratory screen, debt level remains the clearest person-level signal. Ad size and design also show meaningful differences, while partner source and campaign theme remain directionally useful but weaker. Form type and publisher placement do not show strong standalone evidence after the corrected mapping is applied.

I then moved to an adjusted modeling strategy using one common analysis sample. In the staged multivariable comparison, the base model with `month_period`, `DebtLevelGrouped`, and `PartnerClean` achieves the best BIC, while the exploratory LRT screen suggests that adding `CampaignGroup` may provide some incremental signal. Because the criteria do not fully agree, I treat both as candidate models and then check their diagnostics before deciding what to report.

After diagnostics, the simpler base model remains the stronger final reporting model. It captures the most stable adjusted relationships in the data while avoiding the multicollinearity problems that appear when `CampaignGroup` is added on top of `PartnerClean`. The clearest adjusted drivers in this dataset are time and debt segment. `PartnerClean` remains in the model as an important control variable, but its individual coefficients are not statistically significant in the final specification. `CampaignGroup` still provides useful exploratory insight, and state was also reviewed descriptively, but neither adds enough stable adjusted signal to replace debt level as the main person-level predictor in the final model.


#### Limitations and Considerations

Model selection criteria can disagree, especially when one criterion rewards simplicity more heavily than the other. That is why the final adjusted model should not be chosen from BIC or LRT alone.

In this dataset, the biggest remaining risk is overlap between marketing-source variables. `PublisherCampaignName` was already removed because it duplicates `PublisherZoneName`, and the next diagnostics step checks whether `CampaignGroup` also overlaps too strongly with `PartnerClean`. If that overlap is too strong, the LRT-expanded model should be simplified even if it initially looks appealing from a screening perspective.


In [ ]:
final_q2_model = smf.logit(
    "GoodLead ~ C(month_period) + C(DebtLevelGrouped) + C(PartnerClean)",
    data=df_q2_model,
).fit(disp=False)

display(final_q2_model.summary2())


The final logistic regression model examines whether a lead is classified as good (`GoodLead = 1`) as a function of month of lead creation, grouped debt level, and partner source. Overall, the model is statistically significant (`LLR p < 0.001`), indicating that these predictors jointly help explain differences in lead quality. The pseudo R-squared is 0.116, suggesting the model has modest explanatory power, which is reasonable for observational lead-level data.

After adjusting for debt level and partner source, lead quality is significantly lower in every month from May through September 2009 compared with the reference month, April 2009. The decline is especially strong in June and July, where the coefficients are -1.81 and -2.17, respectively (`p < 0.001`), indicating much lower odds of a lead being good during the middle of the period. This is consistent with the earlier time-trend results from Question 1 and suggests that lead quality deteriorated substantially after April.

Debt level is also an important predictor of lead quality. Compared with the reference group (`<=10000`), leads in the `10001-20000`, `20001-50000`, and `50001-90000` groups all have significantly higher odds of being classified as good (`p < 0.001` for all three). This suggests that leads with moderate to high debt balances tend to be higher quality than leads in the lowest debt group. The `>=90000` category is positive but not statistically significant (`p = 0.272`), so there is not enough evidence to conclude that this highest debt group differs from the reference after adjustment.

By contrast, partner source does not appear to be a statistically significant independent driver in the final adjusted model. Although the coefficients for Call Center, Google, and Yahoo are all negative relative to the reference partner, none are statistically significant at the 5% level. This suggests that once month and debt level are controlled for, differences across partner sources are less pronounced.

Overall, the final adjusted model suggests that the strongest stable drivers of lead quality are time and debt level. Lead quality declined significantly after April 2009, and leads with higher debt levels generally had better odds of being classified as good. Partner source was retained as a control variable, but it did not show a statistically significant independent association in the final model.


### Question 2 Scope Notes

I reviewed both `DebtLevelGrouped` and `State` as person-level variables for Question 2. Debt level produced the clearer and more stable signal once the status mapping and debt buckets were corrected. State-level evaluated samples were much more fragmented, so I keep state as descriptive context rather than as a retained adjusted predictor in the final model.


# Question 3: If the advertiser says they will increase our CPL by 20% if we increase lead quality by 20%, do we see any opportunities to do that here? What kinds of things could we do?


Question 3 is less about fitting another model and more about judging whether the economics work. I treat `Closed` as the primary KPI because the assignment notes that the advertiser ultimately makes money when leads close. I still keep the broader `Good` lead rate as a secondary diagnostic because it helps explain where funnel quality is stronger or weaker before the final close step.

For the business case, I use the exact numbers in the prompt, which describe a move from 8.0 percent lead quality to 9.6 percent while CPL rises from $30 to $33. One detail still needs clarification. A move from $30 to $33 is a 10 percent increase in CPL, not a 20 percent increase. That matters because the break-even quality improvement is smaller than the prompt wording suggests.


In [ ]:
q3_prompt_quality = 0.08
q3_prompt_target_quality = 0.096
q3_break_even_quality = 33 / (30 / q3_prompt_quality)

df["ClosedLead"] = (df["CallStatus"] == "Closed").astype(int)
q3_observed_closed_rate = df["ClosedLead"].mean()
q3_observed_good_rate = (df["LeadQuality"] == "Good").mean()

q3_economics = pd.DataFrame({
    "Metric": [
        "Prompt baseline lead quality",
        "Prompt target lead quality",
        "Break-even lead quality at $33 CPL",
        "Observed closed rate in the dataset",
        "Observed broader good lead rate in the dataset",
        "Cost per closed lead at $30 CPL and 8.0% quality",
        "Cost per closed lead at $33 CPL and 8.0% quality",
        "Cost per closed lead at $33 CPL and 9.6% quality",
        "Cost per closed lead at $36 CPL and 9.6% quality",
    ],
    "Value": [
        f"{q3_prompt_quality:.2%}",
        f"{q3_prompt_target_quality:.2%}",
        f"{q3_break_even_quality:.2%}",
        f"{q3_observed_closed_rate:.2%}",
        f"{q3_observed_good_rate:.2%}",
        f"${30 / q3_prompt_quality:,.2f}",
        f"${33 / q3_prompt_quality:,.2f}",
        f"${33 / q3_prompt_target_quality:,.2f}",
        f"${36 / q3_prompt_target_quality:,.2f}",
    ],
})

display(q3_economics)


Using the question exactly as written, the starting lead quality is 8.0 percent and the target is 9.6 percent. At $30 CPL and 8.0 percent quality, cost per closed lead is $375. If CPL rises to $33 and quality improves to 9.6 percent, cost per closed lead falls to about $343.75. On that basis, the proposed deal is economically attractive.

The key nuance is that $30 to $33 is only a 10 percent increase in CPL. Because of that, the break-even quality rate is only 8.8 percent, not 9.6 percent. The dataset also shows an observed closed rate of about 8.1 percent, which is very close to the prompt baseline, and a broader good lead rate of about 13.0 percent. I therefore use `Closed` as the primary business metric for Question 3 and treat the broader `Good` rate as a secondary diagnostic.


In [ ]:
def q3_segment_summary(frame, col):
    out = (
        frame.groupby(col)
        .agg(
            leads=("CallStatus", "size"),
            closed=("ClosedLead", "sum"),
            good=("LeadQuality", lambda s: (s == "Good").sum()),
            bad=("LeadQuality", lambda s: (s == "Bad").sum()),
            unknown=("LeadQuality", lambda s: (s == "Unknown").sum()),
        )
        .reset_index()
    )
    out["closed_rate"] = out["closed"] / out["leads"]
    out["good_rate"] = out["good"] / out["leads"]
    out["unknown_rate"] = out["unknown"] / out["leads"]
    return out.sort_values("closed_rate", ascending=False)

q3_partner = q3_segment_summary(df, "PartnerClean")
q3_campaign = q3_segment_summary(df, "CampaignGroup")
q3_adsize = q3_segment_summary(df, "AdSize")
q3_debt = q3_segment_summary(df, "DebtLevelGrouped")

for name, table in {
    "PartnerClean": q3_partner,
    "CampaignGroup": q3_campaign,
    "AdSize": q3_adsize,
    "DebtLevelGrouped": q3_debt,
}.items():
    print(name)
    display(table.round(4))

q3_partner_campaign = (
    df.groupby(["PartnerClean", "CampaignGroup"])
    .agg(
        leads=("CallStatus", "size"),
        closed=("ClosedLead", "sum"),
        good=("LeadQuality", lambda s: (s == "Good").sum()),
    )
    .reset_index()
)
q3_partner_campaign["closed_rate"] = q3_partner_campaign["closed"] / q3_partner_campaign["leads"]
q3_partner_campaign["good_rate"] = q3_partner_campaign["good"] / q3_partner_campaign["leads"]
q3_partner_campaign = q3_partner_campaign[q3_partner_campaign["leads"] >= 100].sort_values(["closed_rate", "leads"], ascending=[False, False])

display(q3_partner_campaign.round(4))


Using `Closed` as the primary KPI, the segment tables still point to a few clear opportunities. Stronger closed rates appear in AdKnowledge traffic, Credit campaigns, the `300250` ad size, and the middle debt groups. The weakest results appear in Google traffic, Brand campaigns, the `302252` ad size, and the `<=10000` debt group. The broader `Good` rate tells a similar but slightly stronger story, which is why it remains a useful supporting diagnostic.

The interaction view sharpens that signal. Google combined with Brand produces a closed rate of only about 4.0 percent across 629 leads, with a broader good lead rate of 7.0 percent. That is far below the stronger pockets in the data and suggests that the opportunity sits in a few repeated low-quality combinations rather than in one isolated variable.

Some segments still have different rates of unknown outcomes, so any budget shift should be validated with a controlled test before it is scaled. Even so, the closed-rate ranking gives a much cleaner answer to the business question than the earlier broader lead-quality metric alone.


In [ ]:
q3_scenarios = [
    ("Move 20% of Brand campaign volume to Credit campaigns", "CampaignGroup", "Brand", "Credit", 0.20),
    ("Move 20% of Google volume to AdKnowledge closed-rate levels", "PartnerClean", "Google", "AdKnowledge", 0.20),
    ("Move 20% of Google volume to Yahoo closed-rate levels", "PartnerClean", "Google", "Yahoo", 0.20),
    ("Move 20% of 302252 traffic to 300250 traffic", "AdSize", "302252", "300250", 0.20),
    ("Replace 20% of <=10000 debt leads with 10001-20000 debt leads", "DebtLevelGrouped", "<=10000", "10001-20000", 0.20),
    ("Aggressive upper bound: move all Brand volume to Credit campaign closed-rate", "CampaignGroup", "Brand", "Credit", 1.00),
]

rows = []
for label, col, from_seg, to_seg, share in q3_scenarios:
    new_rate = q3_reallocation_scenario(df, col, from_seg, to_seg, share, rate_col="closed_rate")
    rows.append({
        "Scenario": label,
        "New_Closed_Rate_%": round(new_rate * 100, 2),
        "Relative_Lift_vs_Observed_%": round((new_rate / q3_observed_closed_rate - 1) * 100, 2),
        "Cost_per_Closed_Lead_at_$33": round(33 / new_rate, 2),
    })

q3_scenario_results = pd.DataFrame(rows)
display(q3_scenario_results)


The scenario analysis suggests that the opportunity is real, but still not easy. Using observed closed rate as the primary KPI, most 20 percent reallocations improve closed rate by roughly 0 percent to 8 percent on a relative basis. That is helpful, though it suggests that one moderate change by itself is unlikely to guarantee a full 20 percent lift.

The most optimistic case in the table is also the least realistic. If all Brand campaign volume performed like Credit campaigns, closed rate would rise to about 11.2 percent. That is a 38 percent lift relative to the observed closed baseline, but it should be treated as an upper bound rather than a forecast because campaign mix, inventory limits, and partner overlap will constrain how much traffic can really be moved.

The practical takeaway is that the data support improvement, but a full 20 percent lift would likely require several coordinated changes instead of one lever.


### Recommended Actions

1. I would start by cutting back the weakest large pocket, which is Brand traffic coming through weaker partner mixes, especially Google.
2. I would expand Credit-oriented campaigns where scale exists because they show the strongest closed rates in this dataset.
3. I would test a shift away from `302252` inventory and toward `300250`, which performs better on both closed rate and broader good lead rate.
4. I would tighten targeting or qualification for the lowest-debt segment because `<=10000` produces the weakest closed-rate outcome.
5. I would judge these changes primarily on closed rate and cost per closed lead, while keeping the broader `Good` rate as a secondary diagnostic for funnel health.


## Answer to Question 3

Using the question exactly as written, the proposed deal makes economic sense. If lead quality improves from 8.0 percent to 9.6 percent while CPL rises from $30 to $33, cost per closed lead improves from $375 to about $343.75. In fact, because $30 to $33 is only a 10 percent CPL increase, the break-even quality rate is about 8.8 percent. That means the target of 9.6 percent is more than enough to justify the higher CPL.

The more important question is whether the dataset suggests that this kind of improvement is achievable. On the primary business KPI, `Closed`, the answer is yes, but the path looks like optimization through reallocation rather than one isolated tactic. The observed closed rate in the dataset is about 8.1 percent, which is very close to the prompt baseline. The weakest large pockets are Brand campaigns, Google traffic, the `302252` ad size, and the `<=10000` debt segment. The clearest positive pockets are Credit campaigns, the `300250` ad size, and the middle debt groups. The Google plus Brand combination stands out in particular because it carries substantial volume and only about a 4 percent closed rate.

The scenario analysis shows both opportunity and constraint. Most moderate reallocations improve closed rate by only about 0 percent to 8 percent relative from the observed baseline, so a single change is unlikely to deliver the full target on its own. Reaching or exceeding a 20 percent lift looks possible only under aggressive reallocation, and I would treat that as an upper-bound case rather than something I would commit to upfront.

My recommendation would be to run this as a phased optimization program. I would first reduce volume in the weakest large segments, then expand stronger Credit-oriented inventory, test a shift toward better-performing ad sizes, and tighten qualification for the weakest debt segment. I would evaluate each step primarily with closed rate and cost per closed lead, and I would use the broader `Good` rate as a secondary signal for funnel health. So my answer is yes: the economics in the question are attractive, and the dataset does show real opportunities to improve quality, but I would validate the path through controlled tests before promising the full 20 percent lift.


## Key Recommendations

1. Monitor corrected evaluated lead quality over time, but use month-to-month tests that match the observed shape of the data instead of forcing one full-period linear trend.
2. Treat debt segment as the clearest person-level driver, and keep the overlapping `7500-15000` source bucket separate rather than forcing it into a higher debt band.
3. Reallocate budget away from the weakest large pockets, especially Brand traffic paired with weaker partner mixes, and expand stronger Credit-oriented inventory where scale exists.
4. Evaluate every optimization primarily with closed rate and cost per closed lead, and keep the broader `Good` rate as a secondary funnel-health diagnostic.
5. Validate changes through controlled tests before scaling, because the full 20 percent lift target still looks ambitious under realistic reallocations.
